# Urban Mobility Intelligence

## 02 — Exploratory Data Analysis

### Objective

Investigate temporal demand, geographical patterns,
financial behaviour and operational characteristics
within a synthetic urban transportation network.

The analysis aims to identify actionable operational
insights and establish behavioural baselines for
subsequent anomaly detection.

### Data Source

Validated Delta tables created during the data
ingestion and quality assurance stage.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load validated Delta tables

trips_df = spark.table(
    "workspace.default.urban_mobility_trips_validated"
)

zones_df = spark.table(
    "workspace.default.urban_mobility_zones"
)

print("Trip records:", trips_df.count())
print("Reference zones:", zones_df.count())

Trip records: 120000
Reference zones: 20


### 1. Business Performance Overview

Establish baseline operational and financial metrics
to contextualise subsequent demand, geographical
and behavioural analyses.

### Key Performance Indicators

In [0]:
# Calculate business performance indicators

business_overview_df = trips_df.agg(
    F.count("*").alias("total_trips"),

    F.countDistinct("pickup_date").alias("days_observed"),

    F.countDistinct("driver_id").alias("unique_drivers"),

    F.countDistinct("vehicle_id").alias("unique_vehicles"),

    F.round(
        F.sum("total_amount_eur"), 2
    ).alias("total_transaction_value_eur"),

    F.round(
        F.sum("fare_amount_eur"), 2
    ).alias("total_fare_value_eur"),

    F.round(
        F.sum("tip_amount_eur"), 2
    ).alias("total_tips_eur"),

    F.round(
        F.avg("total_amount_eur"), 2
    ).alias("avg_transaction_value_eur"),

    F.round(
        F.sum("trip_distance_km"), 2
    ).alias("total_distance_km"),

    F.round(
        F.avg("trip_distance_km"), 2
    ).alias("avg_distance_km"),

    F.round(
        F.avg("trip_duration_min"), 2
    ).alias("avg_duration_min"),

    F.round(
        F.count("*") / F.countDistinct("pickup_date"), 2
    ).alias("avg_daily_trips")
)

display(business_overview_df)

total_trips,days_observed,unique_drivers,unique_vehicles,total_transaction_value_eur,total_fare_value_eur,total_tips_eur,avg_transaction_value_eur,total_distance_km,avg_distance_km,avg_duration_min,avg_daily_trips
120000,91,300,360,2515430.72,2279367.66,205509.6,20.96,712587.34,5.94,18.34,1318.68


In [0]:
monthly_performance_df = (
    trips_df
    .withColumn(
        "month",
        F.date_format("pickup_date", "yyyy-MM")
    )
    .groupBy("month")
    .agg(
        F.count("*").alias("total_trips"),

        F.countDistinct("pickup_date").alias("days_observed"),

        F.round(
            F.sum("total_amount_eur"), 2
        ).alias("transaction_value_eur"),

        F.round(
            F.avg("total_amount_eur"), 2
        ).alias("avg_transaction_value_eur")
    )
    .withColumn(
        "avg_daily_trips",
        F.round(
            F.col("total_trips") / F.col("days_observed"), 2
        )
    )
    .orderBy("month")
)

display(monthly_performance_df)

month,total_trips,days_observed,transaction_value_eur,avg_transaction_value_eur,avg_daily_trips
2025-09,39686,30,835248.17,21.05,1322.87
2025-10,40848,31,857273.58,20.99,1317.68
2025-11,39466,30,822908.97,20.85,1315.53


#### Hourly Transportation Demand

In [0]:
total_trips = trips_df.count()

hourly_performance_df = (
    trips_df.groupBy("pickup_hour")
    .agg(
        F.count("*").alias("total_trips"),

        F.round(
            F.avg("trip_distance_km"), 2
        ).alias("avg_distance_km"),

        F.round(
            F.avg("trip_duration_min"), 2
        ).alias("avg_duration_min"),

        F.round(
            F.avg("average_speed_kmh"), 2
        ).alias("avg_speed_kmh"),

        F.round(
            F.avg("total_amount_eur"), 2
        ).alias("avg_transaction_value_eur")
    )

    .withColumn(
    "trip_share_pct",
    F.round(
        F.col("total_trips") / F.lit(total_trips) * 100,
        2
    )
)

    .orderBy("pickup_hour")
)

display(hourly_performance_df)

pickup_hour,total_trips,avg_distance_km,avg_duration_min,avg_speed_kmh,avg_transaction_value_eur,trip_share_pct
0,5066,5.96,16.9,23.71,20.5,4.22
1,5024,5.92,16.93,23.52,20.47,4.19
2,5047,5.94,16.98,23.41,20.39,4.21
3,4991,5.85,16.51,23.76,20.31,4.16
4,4925,5.94,16.77,23.52,20.33,4.1
5,5013,6.03,17.17,23.31,20.58,4.18
6,5048,5.9,16.6,23.67,20.25,4.21
7,5002,6.05,22.49,18.29,22.67,4.17
8,5000,5.92,21.89,17.78,22.28,4.17
9,4948,5.97,22.1,17.88,22.37,4.12


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
# Segment trips into operational time periods

time_period_df = trips_df.withColumn(
    "time_period",
    F.when(
        F.col("pickup_hour").between(7, 9),
        "Morning Peak"
    ).when(
        F.col("pickup_hour").between(16, 19),
        "Evening Peak"
    ).otherwise("Off-Peak")
)

period_performance_df = (
    time_period_df
    .groupBy("time_period")
    .agg(
        F.count("*").alias("total_trips"),
        F.countDistinct("pickup_hour").alias("hours_covered"),
        F.round(F.avg("trip_distance_km"), 2)
            .alias("avg_distance_km"),
        F.round(F.avg("trip_duration_min"), 2)
            .alias("avg_duration_min"),
        F.round(F.avg("average_speed_kmh"), 2)
            .alias("avg_speed_kmh"),
        F.round(F.avg("total_amount_eur"), 2)
            .alias("avg_transaction_value_eur"),
        F.round(F.median("average_speed_kmh"), 2)
            .alias("median_speed_kmh")
    )
    .withColumn(
        "avg_trips_per_hour_slot",
        F.round(
            F.col("total_trips") / F.col("hours_covered"),
            2
        )
    )
)

display(period_performance_df)

time_period,total_trips,hours_covered,avg_distance_km,avg_duration_min,avg_speed_kmh,avg_transaction_value_eur,median_speed_kmh,avg_trips_per_hour_slot
Off-Peak,85136,17,5.94,16.84,23.58,20.45,20.99,5008.0
Morning Peak,14950,3,5.98,22.16,17.98,22.44,16.11,4983.33
Evening Peak,19914,4,5.9,21.86,17.93,22.04,16.05,4978.5


#### Temporal Analysis Findings

- Hourly trip volumes remain relatively stable, with each
  hour contributing approximately 4.09%–4.28% of total trips.
- Morning and evening peak-period journeys take 31.6%
  and 29.8% longer than off-peak journeys, respectively.
- Average journey speeds decline by approximately 24%
  during peak periods despite comparable trip distances.
- Average transaction values increase by 9.7% during
  the morning peak and 7.8% during the evening peak.

**Operational implications:**

Longer journey durations may reduce potential trip
throughput despite higher average transaction values.

**Machine learning implications:**

Temporal context should be considered when establishing
expected journey behaviour and identifying anomalies.

**Dataset limitation:**

The synthetic dataset exhibits congestion-related
variation but limited temporal variation in trip demand.
Peak periods therefore represent simulated congestion
periods rather than observed demand peaks.

### 3. Geographical Demand and Movement Patterns

Investigate spatial demand distribution, origin-destination
patterns and geographical differences in journey
characteristics using enriched zone reference information.

In [0]:
# Prepare pickup zone reference

pickup_zones_df = zones_df.select(
    F.col("zone_name").alias("pickup_zone"),
    F.col("zone_type").alias("pickup_zone_type"),
    F.col("area_group").alias("pickup_area_group")
)

# Prepare drop-off zone reference

dropoff_zones_df = zones_df.select(
    F.col("zone_name").alias("dropoff_zone"),
    F.col("zone_type").alias("dropoff_zone_type"),
    F.col("area_group").alias("dropoff_area_group")
)

# Enrich trip records with geographical attributes

enriched_trips_df = (
    trips_df
    .join(pickup_zones_df, "pickup_zone", "left")
    .join(dropoff_zones_df, "dropoff_zone", "left")
)

print("Enriched records:", enriched_trips_df.count())
print("Total columns:", len(enriched_trips_df.columns))

Enriched records: 120000
Total columns: 30


In [0]:
# Analyse demand and financial activity by pickup zone

pickup_performance_df = (
    enriched_trips_df
    .groupBy(
        "pickup_zone",
        "pickup_zone_type",
        "pickup_area_group"
    )
    .agg(
        F.count("*").alias("total_trips"),

        F.round(F.sum("total_amount_eur"), 2)
            .alias("transaction_value_eur"),

        F.round(F.avg("total_amount_eur"), 2)
            .alias("avg_transaction_value_eur"),

        F.round(F.avg("trip_distance_km"), 2)
            .alias("avg_distance_km"),

        F.round(F.avg("trip_duration_min"), 2)
            .alias("avg_duration_min")
    )
    .withColumn(
        "trip_share_pct",
        F.round(
            F.col("total_trips") / F.lit(total_trips) * 100,
            2
        )
    )
    .orderBy(F.desc("total_trips"))
)

display(pickup_performance_df)

pickup_zone,pickup_zone_type,pickup_area_group,total_trips,transaction_value_eur,avg_transaction_value_eur,avg_distance_km,avg_duration_min,trip_share_pct
Central Business District,Business,Core,15544,320630.06,20.63,5.86,18.1,12.95
Central Station,Transport,Core,12042,243980.79,20.26,5.74,17.86,10.04
Airport,Transport,Outer,8357,242165.14,28.98,8.27,24.43,6.96
Financial Quarter,Business,Core,8338,171322.42,20.55,5.79,18.01,6.95
North Residential,Residential,North,8330,172202.28,20.67,5.84,17.98,6.94
University District,Education,Inner,7155,143474.78,20.05,5.68,17.74,5.96
South Residential,Residential,South,7061,143237.84,20.29,5.77,17.9,5.88
West Shopping District,Retail,West,7030,143840.12,20.46,5.83,18.02,5.86
Old Town,Tourism,Core,4761,96951.27,20.36,5.78,17.9,3.97
Harbourfront,Leisure,West,4754,95789.61,20.15,5.64,17.46,3.96


Databricks visualization. Run in Databricks to view.

In [0]:
# Count outgoing trips

outgoing_df = (
    trips_df.groupBy("pickup_zone")
    .agg(F.count("*").alias("outgoing_trips"))
    .withColumnRenamed("pickup_zone", "zone_name")
)

# Count incoming trips

incoming_df = (
    trips_df.groupBy("dropoff_zone")
    .agg(F.count("*").alias("incoming_trips"))
    .withColumnRenamed("dropoff_zone", "zone_name")
)

# Compare incoming and outgoing trip activity

zone_balance_df = (
    zones_df
    .join(outgoing_df, "zone_name", "left")
    .join(incoming_df, "zone_name", "left")
    .fillna(0, subset=["incoming_trips", "outgoing_trips"])
    .withColumn(
        "net_trip_flow",
        F.col("incoming_trips") - F.col("outgoing_trips")
    )
    .withColumn(
        "absolute_imbalance",
        F.abs(F.col("net_trip_flow"))
    )
    .orderBy(F.desc("absolute_imbalance"))
)

display(zone_balance_df)

zone_name,zone_type,area_group,outgoing_trips,incoming_trips,net_trip_flow,absolute_imbalance
Central Business District,Business,Core,15544,16607,1063,1063
Central Station,Transport,Core,12042,11400,-642,642
University District,Education,Inner,7155,6886,-269,269
Airport,Transport,Outer,8357,8197,-160,160
Technology Park,Business,East,4616,4776,160,160
Financial Quarter,Business,Core,8338,8188,-150,150
South Residential,Residential,South,7061,7207,146,146
Harbourfront,Leisure,West,4754,4638,-116,116
Old Town,Tourism,Core,4761,4645,-116,116
East Medical Campus,Healthcare,East,4742,4852,110,110


In [0]:
# Identify frequently travelled origin-destination pairs

route_performance_df = (
    enriched_trips_df
    .groupBy("pickup_zone", "dropoff_zone")
    .agg(
        F.count("*").alias("total_trips"),

        F.round(F.avg("trip_distance_km"), 2)
            .alias("avg_distance_km"),

        F.round(F.avg("trip_duration_min"), 2)
            .alias("avg_duration_min"),

        F.round(F.avg("total_amount_eur"), 2)
            .alias("avg_transaction_value_eur"),

        F.round(F.median("fare_per_km_eur"), 2)
            .alias("median_fare_per_km")
    )
    .orderBy(F.desc("total_trips"))
)

display(route_performance_df.limit(15))

pickup_zone,dropoff_zone,total_trips,avg_distance_km,avg_duration_min,avg_transaction_value_eur,median_fare_per_km
Central Station,Central Business District,2563,5.58,17.37,19.65,3.44
Airport,Central Business District,2234,8.05,23.89,28.26,3.16
Central Business District,Central Station,1734,5.75,17.7,19.8,3.41
Central Business District,Airport,1214,8.41,24.46,29.22,3.14
Central Business District,Financial Quarter,1178,5.52,17.54,19.72,3.47
Central Business District,North Residential,1177,5.57,17.42,19.73,3.48
Financial Quarter,Central Business District,1129,5.61,17.32,19.78,3.44
North Residential,Central Business District,1127,5.84,17.54,20.38,3.44
Central Business District,South Residential,1094,5.62,18.06,19.83,3.46
Central Business District,University District,1022,5.76,17.48,19.76,3.44


In [0]:
# Prepare all origin-destination connections

all_routes_df = (
    route_performance_df
    .select(
        F.col("pickup_zone").alias("Origin"),
        F.col("dropoff_zone").alias("Destination"),
        F.col("total_trips").alias("Trips")
    )
)

display(all_routes_df)

Origin,Destination,Trips
Central Station,Central Business District,2563
Airport,Central Business District,2234
Central Business District,Central Station,1734
Central Business District,Airport,1214
Central Business District,Financial Quarter,1178
Central Business District,North Residential,1177
Financial Quarter,Central Business District,1129
North Residential,Central Business District,1127
Central Business District,South Residential,1094
Central Business District,University District,1022


Databricks visualization. Run in Databricks to view.

In [0]:
# Prepare the 15 busiest origin-destination routes

top_15_routes_df = (
    route_performance_df
    .withColumn(
        "route",
        F.concat_ws(
            " → ",
            F.regexp_replace(
                F.regexp_replace(
                    F.col("pickup_zone"),
                    "Central Business District", "CBD"
                ),
                "Central Station", "Station"
            ),
            F.regexp_replace(
                F.regexp_replace(
                    F.col("dropoff_zone"),
                    "Central Business District", "CBD"
                ),
                "Central Station", "Station"
            )
        )
    )
    .select("route", "total_trips")
    .orderBy(F.desc("total_trips"))
    .limit(15)
)

display(top_15_routes_df)

route,total_trips
Station → CBD,2563
Airport → CBD,2234
CBD → Station,1734
CBD → Airport,1214
CBD → Financial Quarter,1178
CBD → North Residential,1177
Financial Quarter → CBD,1129
North Residential → CBD,1127
CBD → South Residential,1094
CBD → University District,1022


Databricks visualization. Run in Databricks to view.

#### Geographical Analysis Findings

- The Central Business District and Central Station account
  for approximately 23% of all recorded pickups.
- Airport-originating journeys have an average transaction
  value of €28.98, approximately 38% above the overall
  average, alongside longer distances and durations.
- The Central Business District records a net inflow of
  1,063 trips, while Central Station records a net outflow
  of 642 trips.
- Central Station to Central Business District is the
  most frequently recorded route, with 2,563 journeys.

**Operational implications:**

Central zones represent important concentrations of
transportation activity. Directional trip imbalances
could inform further fleet-positioning investigations.

**Anomaly detection implications:**

Trip origin, destination and route context may help
distinguish expected high-value journeys from unusual
transaction behaviour.

**Limitations:**

All locations are fictional. Trip flows do not establish
actual vehicle availability, and no real geographic
distances or road networks are represented.

In [0]:
# Compare financial characteristics across business segments

segment_results = []

for segment in [
    "payment_method",
    "service_type",
    "dispatch_channel"
]:
    result = (
        trips_df
        .groupBy(segment)
        .agg(
            F.count("*").alias("total_trips"),
            F.round(F.avg("fare_amount_eur"), 2)
                .alias("avg_fare_eur"),
            F.round(F.avg("total_amount_eur"), 2)
                .alias("avg_transaction_eur"),
            F.round(F.expr(
                "percentile_approx(total_amount_eur, 0.5)"
            ), 2).alias("median_transaction_eur"),
            F.round(F.avg("trip_distance_km"), 2)
                .alias("avg_distance_km"),
            F.round(F.avg("trip_duration_min"), 2)
                .alias("avg_duration_min")
        )
        .withColumnRenamed(segment, "segment_value")
        .withColumn("segment_type", F.lit(segment))
    )

    segment_results.append(result)

financial_segments_df = segment_results[0]

for result in segment_results[1:]:
    financial_segments_df = financial_segments_df.unionByName(result)

display(
    financial_segments_df.orderBy(
        "segment_type",
        F.desc("total_trips")
    )
)

segment_value,total_trips,avg_fare_eur,avg_transaction_eur,median_transaction_eur,avg_distance_km,avg_duration_min,segment_type
Mobile App,75289,19.07,21.04,16.86,5.96,18.39,dispatch_channel
Street Hail,21765,18.92,20.92,16.77,5.92,18.21,dispatch_channel
Corporate Booking,13190,18.79,20.71,16.67,5.87,18.28,dispatch_channel
Phone Dispatch,9756,18.88,20.79,16.93,5.94,18.31,dispatch_channel
Card,74304,18.99,20.95,16.83,5.94,18.32,payment_method
Cash,23948,19.01,21.01,16.88,5.94,18.42,payment_method
Mobile Wallet,16913,19.09,21.05,16.82,5.97,18.38,payment_method
Corporate Account,4835,18.62,20.57,16.48,5.78,18.07,payment_method
Standard,105608,18.98,20.95,16.83,5.93,18.33,service_type
Accessible,8395,19.06,21.02,16.59,5.93,18.15,service_type


In [0]:
# Convert the small aggregated result to Pandas
financial_segments_pd = financial_segments_df.toPandas()

# Define colours for each segment
segment_colors = {
    "payment_method": "#DBEAFE",    # Light blue
    "service_type": "#D1FAE5",      # Light green
    "dispatch_channel": "#FEF3C7"   # Light orange
}

# Apply background colours to each row
def highlight_segment(row):
    color = segment_colors.get(
        row["segment_type"], "#FFFFFF"
    )
    return [f"background-color: {color}"] * len(row)

# Display styled results
display(
    financial_segments_pd.style.apply(
        highlight_segment, axis=1
    )
)

,segment_value,total_trips,avg_fare_eur,avg_transaction_eur,median_transaction_eur,avg_distance_km,avg_duration_min,segment_type
0,Corporate Account,4835,18.620000,20.570000,16.480000,5.780000,18.070000,payment_method
1,Card,74304,18.990000,20.950000,16.830000,5.940000,18.320000,payment_method
2,Mobile Wallet,16913,19.090000,21.050000,16.820000,5.970000,18.380000,payment_method
3,Cash,23948,19.010000,21.010000,16.880000,5.940000,18.420000,payment_method
4,Premium,5997,19.200000,21.130000,17.120000,6.040000,18.740000,service_type
5,Accessible,8395,19.060000,21.020000,16.590000,5.930000,18.150000,service_type
6,Standard,105608,18.980000,20.950000,16.830000,5.930000,18.330000,service_type
7,Mobile App,75289,19.070000,21.040000,16.860000,5.960000,18.390000,dispatch_channel
8,Corporate Booking,13190,18.790000,20.710000,16.670000,5.870000,18.280000,dispatch_channel
9,Phone Dispatch,9756,18.880000,20.790000,16.930000,5.940000,18.310000,dispatch_channel


In [0]:
financial_chart_df = (
    financial_segments_df
    .withColumn(
        "Payment Method",
        F.when(
            F.col("segment_type") == "payment_method",
            F.col("total_trips")
        )
    )
    .withColumn(
        "Service Type",
        F.when(
            F.col("segment_type") == "service_type",
            F.col("total_trips")
        )
    )
    .withColumn(
        "Dispatch Channel",
        F.when(
            F.col("segment_type") == "dispatch_channel",
            F.col("total_trips")
        )
    )
    .select(
        "segment_value",
        "Payment Method",
        "Service Type",
        "Dispatch Channel"
    )
)

display(financial_chart_df)

segment_value,Payment Method,Service Type,Dispatch Channel
Corporate Account,4835,null,null
Card,74304,null,null
Mobile Wallet,16913,null,null
Cash,23948,null,null
Premium,null,5997,null
Accessible,null,8395,null
Standard,null,105608,null
Mobile App,null,null,75289
Corporate Booking,null,null,13190
Phone Dispatch,null,null,9756


Databricks visualization. Run in Databricks to view.

In [0]:
# Examine financial distribution and extreme transaction values

transaction_distribution_df = trips_df.agg(
    F.round(F.avg("total_amount_eur"), 2)
        .alias("mean_transaction_eur"),

    F.round(F.median("total_amount_eur"), 2)
        .alias("median_transaction_eur"),

    F.round(
        F.percentile_approx(
            "total_amount_eur", 0.95
        ), 2
    ).alias("p95_transaction_eur"),

    F.round(
        F.percentile_approx(
            "total_amount_eur", 0.99
        ), 2
    ).alias("p99_transaction_eur"),

    F.round(F.max("total_amount_eur"), 2)
        .alias("max_transaction_eur")
)

display(transaction_distribution_df)

mean_transaction_eur,median_transaction_eur,p95_transaction_eur,p99_transaction_eur,max_transaction_eur
20.96,16.83,47.76,78.66,459.52


### 4.2 Transaction Distribution Findings

- The average transaction value is €20.96, compared with
  a median of €16.83, indicating a right-skewed distribution.
- The 95th and 99th percentile transaction values are
  approximately €47.76 and €78.66, respectively.
- The highest recorded transaction is €459.52.

**Analytical implications:**

Transaction values exhibit a substantial upper tail.
High-value journeys should be evaluated in relation to
distance, duration, service type and geographical context
rather than treated as anomalous solely because of
their monetary value.

**Anomaly detection implications:**

Both absolute transaction values and contextual pricing
features will be considered during feature engineering.

## EDA Summary and Modelling Implications

Exploratory analysis of 120,000 synthetic urban mobility trips
identified several patterns relevant to anomaly detection.

### Key findings

1. Trip activity is relatively uniform across hours, while
   simulated peak periods exhibit slower speeds and longer
   journey durations.

2. Transaction values are right-skewed, with a median of
   €16.83, a 99th percentile of €78.66 and a maximum of €459.52.

3. Average transaction values differ only modestly across
   payment methods, service types and dispatch channels.

4. Median speed, duration and fare per kilometre differ
   between peak and off-peak periods.

### Implications for feature engineering

- Retain journey distance, duration, speed and pricing measures.
- Create contextual deviations relative to operational periods.
- Retain relevant categorical attributes for potential analysis.
- Avoid treating high transaction values as automatically anomalous.
- Exclude synthetic scenario labels from model features.

All findings describe a synthetic dataset and should not be
interpreted as observed behaviour in a real transportation network.